<a href="https://colab.research.google.com/github/InahH-004/Parking-Space-Detection/blob/Farhan/Tahap_4_Pelatihan_%26_Evaluasi_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tahap 4: Pelatihan, Optimasi, dan Evaluasi Model Machine Learning

**Topik Proyek:** Pendeteksi Ketersediaan Slot Parkir Mobil (Pengolahan Citra Digital + Classical ML)  
**Peran:** Anggota 4 — ML Modeler  
**Output Utama:** `models/best_model.pkl`

---

## Ringkasan Alur Pengerjaan
Pada tahap ini, vektor fitur numerik hasil ekstraksi Tahap 3 (`HOG + Spatial LBP + Warna HSV/LAB`) digunakan untuk melatih model klasifikasi biner **Machine Learning klasik (non-Deep Learning)** guna membedakan status slot parkir:
* **`0` (Kosong / Free):** Tidak terdapat kendaraan pada petak parkir.
* **`1` (Terisi / Occupied):** Terdapat kendaraan penuh atau sebagian pada petak parkir.

Tahapan yang dilakukan meliputi:
1. Memuat matriks fitur numerik dari file `./data/features/fitur_pcd_hog_lbp_warna.npz`.
2. Membangun *baseline model* menggunakan **SVM**, **Random Forest**, dan **LightGBM**.
3. Melakukan *hyperparameter tuning* dengan **5-Fold Stratified Cross-Validation (`GridSearchCV`)**.
4. Mengevaluasi performa akhir pada *Test Set* menggunakan metrik **Accuracy, Precision, Recall, F1-Score**, dan **Confusion Matrix**.
5. Mengekspor *pipeline* model terbaik ke format `.pkl` agar siap dipakai pada Tahap 5 (Inferensi & UI).

## 0. Sinkronisasi Repositori & Generasi Fitur Tahap 3
Karena file dataset mentah dan hasil ekstraksi fitur tidak disimpan di repositori GitHub, sel ini menarik kode terbaru dari repositori tim dan menjalankan notebook Tahap 3 untuk menghasilkan file `./data/features/fitur_pcd_hog_lbp_warna.npz` pada runtime Colab saat ini.

In [ ]:
# 1. Clone repo (ganti 'main' dengan nama branch temanmu jika Tahap 3 belum di-merge ke main)
!git clone -b Fauzan https://github.com/InahH-004/Parking-Space-Detection.git
%cd Parking-Space-Detection

# 2. Jalankan notebook Tahap 3 secara otomatis untuk menghasilkan ./data/features/fitur_pcd_hog_lbp_warna.npz
# (Pastikan nama file sesuai dengan yang ada di GitHub)
!jupyter nbconvert --to notebook --execute Tahap_3_Ekstraksi_Fitur.ipynb --output /tmp/executed_tahap3.ipynb

## 1. Import Library & Memuat Dataset Fitur
Pada bagian ini, kita memuat matriks fitur mentah (`X_train_raw`, `X_val_raw`, `X_test_raw`) beserta label biner (`y_train`, `y_val`, `y_test`) dari file `.npz` hasil Tahap 3.

> **Catatan Teknis (Pencegahan Data Leakage & Integrasi Tahap 5):**  
> Kita sengaja memuat fitur *raw* (belum di-*scale*) dan membungkus `StandardScaler()` di dalam `sklearn.pipeline.Pipeline`. Tujuannya agar proses standarisasi dihitung secara terpisah di tiap *fold* saat *cross-validation*, serta otomatis ikut tersimpan di dalam file `best_model.pkl` sehingga Tahap 5 dapat langsung memprediksi vektor fitur baru tanpa perlu *scaling* manual.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

RANDOM_STATE = 42

# Load file .npz hasil Tahap 3
features_path = "./data/features/fitur_pcd_hog_lbp_warna.npz"
assert os.path.exists(features_path), f"File {features_path} tidak ditemukan! Pastikan Tahap 3 sudah dijalankan."

data = np.load(features_path)

# Gunakan data RAW agar StandardScaler bisa masuk ke dalam Pipeline model
X_train_raw = data["X_train_raw"]
X_val_raw   = data["X_val_raw"]
X_test_raw  = data["X_test_raw"]

y_train = data["y_train"]
y_val   = data["y_val"]
y_test  = data["y_test"]

print("=== DATA FITUR BERHASIL DIMUAT ===")
print(f"Train : {X_train_raw.shape} | Distribusi [0, 1]: {np.bincount(y_train)}")
print(f"Val   : {X_val_raw.shape}   | Distribusi [0, 1]: {np.bincount(y_val)}")
print(f"Test  : {X_test_raw.shape}  | Distribusi [0, 1]: {np.bincount(y_test)}")

## 2. Baseline Modeling (Perbandingan Awal 3 Algoritma)
Sebelum melakukan optimasi parameter, ketiga kandidat algoritma Machine Learning klasik dilatih menggunakan konfigurasi dasar (*default/baseline*) pada **Train Set**, lalu dievaluasi pada **Validation Set**:
* **Support Vector Machine (SVM - RBF Kernel):** Sangat efektif untuk data berdimensi tinggi seperti vektor fitur HOG + LBP.
* **Random Forest:** Algoritma *ensemble bagging* berbasis pohon keputusan yang tahan terhadap *noise* dan *overfitting*.
* **LightGBM:** Algoritma *gradient boosting* yang cepat dan kuat dalam menangkap pola non-linear pada data tabular numerik.

In [ ]:
baseline_models = {
    "SVM (RBF)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1))
    ]),
    "LightGBM": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LGBMClassifier(n_estimators=200, random_state=RANDOM_STATE, verbose=-1))
    ])
}

baseline_results = []
for name, pipe in baseline_models.items():
    pipe.fit(X_train_raw, y_train)
    y_val_pred = pipe.predict(X_val_raw)

    baseline_results.append({
        "Model": name,
        "Val Accuracy": accuracy_score(y_val, y_val_pred),
        "Val Precision": precision_score(y_val, y_val_pred),
        "Val Recall": recall_score(y_val, y_val_pred),
        "Val F1-Score": f1_score(y_val, y_val_pred)
    })

df_baseline = pd.DataFrame(baseline_results).sort_values(by="Val F1-Score", ascending=False)
print("=== PERBANDINGAN BASELINE MODEL (VALIDATION SET) ===")
display(df_baseline)

## 3. Hyperparameter Tuning (5-Fold Stratified Cross-Validation)
Untuk mendapatkan performa optimal sekaligus menjaga stabilitas generalisasi model, dilakukan pencarian kombinasi parameter terbaik menggunakan `GridSearchCV` dengan **5-Fold Stratified Cross-Validation**.

* **Stratified K-Fold** memastikan proporsi kelas `0` (Kosong) dan `1` (Terisi) tetap seimbang pada setiap *fold*.
* **Metrik Utama (`F1-Score`):** Dipilih sebagai acuan optimasi agar model menyeimbangkan *Precision* (minim salah tebak slot terisi) dan *Recall* (minim melewatkan kendaraan yang terparkir).
* Model dengan **Validation F1-Score** tertinggi akan dipilih sebagai model final.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

param_grids = {
    "SVM": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(probability=True, random_state=RANDOM_STATE))
        ]),
        "params": {
            "clf__C": [0.1, 1, 10, 50],
            "clf__gamma": ["scale", "auto", 0.01, 0.001],
            "clf__kernel": ["rbf", "linear"]
        }
    },
    "Random Forest": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
        ]),
        "params": {
            "clf__n_estimators": [100, 200, 300],
            "clf__max_depth": [None, 10, 20],
            "clf__min_samples_split": [2, 5]
        }
    },
    "LightGBM": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LGBMClassifier(random_state=RANDOM_STATE, verbose=-1))
        ]),
        "params": {
            "clf__n_estimators": [100, 200],
            "clf__learning_rate": [0.05, 0.1],
            "clf__num_leaves": [15, 31]
        }
    }
}

tuned_results = []
tuned_models = {}

for name, cfg in param_grids.items():
    print(f"Sedang melakukan GridSearchCV untuk {name}...")
    grid = GridSearchCV(
        estimator=cfg["pipeline"],
        param_grid=cfg["params"],
        cv=cv,
        scoring="f1",
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_train_raw, y_train)

    # Evaluasi kandidat terbaik pada Validation Set
    best_est = grid.best_estimator_
    y_val_pred = best_est.predict(X_val_raw)
    val_f1 = f1_score(y_val, y_val_pred)

    tuned_models[name] = best_est
    tuned_results.append({
        "Model": name,
        "Best CV F1": grid.best_score_,
        "Val Accuracy": accuracy_score(y_val, y_val_pred),
        "Val F1-Score": val_f1,
        "Best Params": grid.best_params_
    })

df_tuned = pd.DataFrame(tuned_results).sort_values(by="Val F1-Score", ascending=False)
print("\n=== HASIL HYPERPARAMETER TUNING (VALIDATION SET) ===")
display(df_tuned[["Model", "Best CV F1", "Val Accuracy", "Val F1-Score"]])

# Pilih model terbaik berdasarkan Val F1-Score
best_model_name = df_tuned.iloc[0]["Model"]
best_model = tuned_models[best_model_name]
print(f"\nModel Terpilih: {best_model_name}")
print(f"Parameter Terbaik: {df_tuned.iloc[0]['Best Params']}")

## 4. Evaluasi Final pada Test Set & Confusion Matrix
Model terbaik hasil *tuning* diuji secara objektif pada **Test Set** (data yang sama sekali belum pernah dilihat selama pelatihan maupun *tuning*). Evaluasi mencakup:
* **Accuracy:** Persentase keseluruhan prediksi yang benar.
* **Precision & Recall:** Ketepatan deteksi pada masing-masing kelas.
* **Confusion Matrix:** Memetakan jumlah *True Negative* (Kosong terdeteksi Kosong), *True Positive* (Terisi terdeteksi Terisi), serta kesalahan prediksi (*False Positive* dan *False Negative*).

In [ ]:
y_test_pred = best_model.predict(X_test_raw)

print(f"=== EVALUASI FINAL PADA TEST SET ({best_model_name}) ===")
print(f"Test Accuracy  : {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Test Precision : {precision_score(y_test, y_test_pred):.4f}")
print(f"Test Recall    : {recall_score(y_test, y_test_pred):.4f}")
print(f"Test F1-Score  : {f1_score(y_test, y_test_pred):.4f}\n")

print(classification_report(y_test, y_test_pred, target_names=["0: Kosong", "1: Terisi"]))

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["0: Kosong", "1: Terisi"],
    yticklabels=["0: Kosong", "1: Terisi"]
)
plt.title(f"Confusion Matrix (Test Set) - {best_model_name}", fontsize=13, fontweight="bold")
plt.xlabel("Prediksi Model")
plt.ylabel("Label Aktual")
plt.tight_layout()
plt.show()

## 5. Export Model Terbaik (`models/best_model.pkl`) & Sanity Check
Model *pipeline* terbaik (`StandardScaler` + *Classifier*) diserialisasi menggunakan `joblib` ke direktori `models/best_model.pkl`.  
Pada sel ini juga dilakukan *sanity check* dengan memuat kembali file `.pkl` tersebut dan menjalankan simulasi prediksi untuk 1 sampel fitur mentah guna memastikan model siap diintegrasikan pada **Tahap 5**.

In [ ]:
os.makedirs("models", exist_ok=True)
model_save_path = "models/best_model.pkl"

joblib.dump(best_model, model_save_path)
print(f"Model terbaik ({best_model_name}) berhasil diekspor ke: {model_save_path}")

# Sanity Check: simulasi pemanggilan di Tahap 5 (1 sampel raw feature)
loaded_model = joblib.load(model_save_path)
sample_pred = loaded_model.predict(X_test_raw[:1])[0]
sample_prob = loaded_model.predict_proba(X_test_raw[:1])[0]
print(f"Uji Prediksi 1 Sampel -> Kelas: {sample_pred} | Probabilitas [Kosong, Terisi]: {sample_prob.round(4)}")

## 6. Kesimpulan & Panduan Integrasi untuk Tahap 5

### Ringkasan Hasil Eksperimen
* **Model Terbaik:** LightGBM (`learning_rate=0.1`, `n_estimators=200`, `num_leaves=31`)
* **Performa Cross-Validation & Validasi:**
  * **Best 5-Fold CV F1-Score:** `0.9784`
  * **Validation Accuracy:** `98.52%` (`0.9852`)
  * **Validation F1-Score:** `0.9895`
* **Performa Evaluasi Final (Test Set):**
  * **Test Accuracy:** `[Isi angka dari Sel 4, misal: 97.78%]`
  * **Test F1-Score:** `[Isi angka dari Sel 4, misal: 0.9815]`

### Analisis Perbandingan Model
1. **LightGBM vs. SVM:** Kedua model menunjukkan performa yang sangat kompetitif dengan *Validation Accuracy* identik (`98.52%`). Meskipun SVM mencatat *CV F1-Score* sedikit lebih tinggi (`0.9794` vs. `0.9784`), **LightGBM** unggul tipis pada *Validation F1-Score* (`0.9895` vs. `0.9892`). Selain itu, LightGBM memiliki waktu inferensi yang sangat cepat sehingga ideal untuk memprediksi puluhan slot parkir secara simultan pada Tahap 5.
2. **Perbandingan dengan Random Forest:** Random Forest mencatat *Validation Accuracy* `96.30%` dan *Validation F1-Score* `0.9741`. Selisih ini menunjukkan bahwa kombinasi vektor fitur berdimensi tinggi (`HOG + Spatial LBP + Warna HSV/LAB`) lebih optimal dipisahkan menggunakan pendekatan *gradient boosting* (LightGBM) atau *kernel margin* (SVM) dibandingkan *bagging* standar.
3. **Kesimpulan Fitur PCD:** Tingginya skor F1 pada ketiga algoritma (> `0.97`) membuktikan bahwa ekstraksi fitur PCD murni tanpa Deep Learning sudah sangat diskriminatif untuk membedakan slot parkir kosong dan terisi.

### Cara Memanggil Model di Tahap 5 (`05_full_pipeline_inference.ipynb`)
```python
import joblib

# 1. Load model pipeline (sudah termasuk StandardScaler + LightGBM)
model = joblib.load("models/best_model.pkl")

# 2. Ekstrak fitur dari 1 patch citra (BGR) menggunakan fungsi Tahap 3
vektor_fitur, _ = ekstrak_vektor_fitur_pcd(patch_bgr)

# 3. Prediksi kelas (0 = Kosong, 1 = Terisi)
prediksi = model.predict(vektor_fitur.reshape(1, -1))[0]
probabilitas = model.predict_proba(vektor_fitur.reshape(1, -1))[0]